# KYC Thesis — Model 1: Document Quality Classifier

**What this notebook does, in order:**
1. Installs dependencies
2. Generates a synthetic training dataset (5 quality classes)
3. Trains MobileNetV3-Small with transfer learning
4. Evaluates with full metrics + confusion matrix
5. Runs the compression study (FP32 → INT8 PTQ → Distilled)
6. Exports to TFLite for Flutter
7. Saves everything to Google Drive

**Runtime:** ~2 hours on Colab T4 GPU

**Before running:** Runtime → Change runtime type → T4 GPU

## Cell 1 — Install Dependencies

In [ ]:
%%capture
!pip install 
numpy==1.26.4 pandas==2.2.2 
timm==0.9.16 albumentations==1.3.1 mlflow==2.10.0 
onnxruntime==1.17.0 faker
!pip install onnx-tf tensorflow  # for TFLite export

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## Cell 2 — Imports

In [ ]:
import os
import json
import time
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import cv2
from faker import Faker

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)
import mlflow
import mlflow.pytorch

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# Class definitions — order matters, used everywhere
CLASSES     = ['GOOD', 'BLURRY', 'GLARE', 'DARK', 'NO_DOCUMENT']
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
print(f'Classes: {CLASS_TO_IDX}')


## Cell 3 — Mount Google Drive

All outputs (models, dataset, results) are saved here so nothing is lost when Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project directory structure in Drive
BASE_DIR     = Path('/content/drive/MyDrive/kyc_thesis')
DATA_DIR     = BASE_DIR / 'data' / 'quality_dataset'
CHECKPOINT_DIR = BASE_DIR / 'models' / 'checkpoints'
EXPORT_DIR   = BASE_DIR / 'models' / 'exports'
RESULTS_DIR  = BASE_DIR / 'results' / 'quality_model'

for d in [DATA_DIR, CHECKPOINT_DIR, EXPORT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directory structure created:')
print(f'  Data:        {DATA_DIR}')
print(f'  Checkpoints: {CHECKPOINT_DIR}')
print(f'  Exports:     {EXPORT_DIR}')
print(f'  Results:     {RESULTS_DIR}')

## Cell 4 — Dataset Generator

**How this works:**
- Generate a clean synthetic ID document image as the base
- Apply a class-specific augmentation to create each quality label
- GOOD = light augmentation only
- BLURRY = heavy Gaussian blur
- GLARE = bright white patch + high contrast
- DARK = heavy brightness reduction
- NO_DOCUMENT = solid color patches (no document content)

**Key design decision:** augmentations are AGGRESSIVE, not borderline.
Clear class separation during training → confident predictions at inference time.

In [ ]:
fake = Faker()

# ── Font helpers ─────────────────────────────────────────────────────
FONT_CANDIDATES_REG = [
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf'
]
FONT_CANDIDATES_BOLD = [
    '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf'
]
FONT_CANDIDATES_MRZ = [
    '/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf'
]

def _load_font(candidates, size):
    for path in candidates:
        if Path(path).exists():
            try:
                return ImageFont.truetype(path, size)
            except Exception:
                continue
    return ImageFont.load_default()

FONT_REG   = _load_font(FONT_CANDIDATES_REG, 14)
FONT_BOLD  = _load_font(FONT_CANDIDATES_BOLD, 15)
FONT_SMALL = _load_font(FONT_CANDIDATES_REG, 12)
FONT_MRZ   = _load_font(FONT_CANDIDATES_MRZ, 14)

# ── Template layouts ────────────────────────────────────────────────
TEMPLATES = [
    {
        'bg': (235, 240, 255),
        'text': (20, 20, 60),
        'border': (80, 100, 180),
        'header_text': 'NATIONAL IDENTITY CARD',
        'label_color': (90, 90, 90),
        'photo_box': [15, 55, 95, 145],
        'field_x': 110,
        'value_x': 190,
        'field_y': 60,
        'field_step': 18,
        'mrz_y': 168,
    },
    {
        'bg': (255, 245, 230),
        'text': (40, 20, 10),
        'border': (180, 120, 60),
        'header_text': 'IDENTITY CARD',
        'label_color': (110, 100, 90),
        'photo_box': [18, 60, 100, 150],
        'field_x': 115,
        'value_x': 195,
        'field_y': 65,
        'field_step': 18,
        'mrz_y': 170,
    },
]


def _synthetic_fields():
    return {
        'SURNAME': fake.last_name().upper(),
        'GIVEN NAMES': fake.first_name().upper() + ' ' + fake.last_name().upper(),
        'ID NO.': f'{random.randint(10000000, 99999999)}',
        'D.O.B.': fake.date_of_birth(minimum_age=18, maximum_age=70).strftime('%d/%m/%Y'),
        'EXPIRY': fake.future_date(end_date='+10y').strftime('%d/%m/%Y'),
        'NATION.': fake.country().upper()[:15],
    }


def _apply_paper_texture(img, strength=5):
    arr = np.array(img).astype(np.int16)
    noise = np.random.normal(0, strength, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


def render_base_document(size=(400, 260)):
    """Render a clean synthetic ID card image with a structured layout."""
    tpl = random.choice(TEMPLATES)
    img = Image.new('RGB', size, tpl['bg'])
    draw = ImageDraw.Draw(img)

    # Border + header
    draw.rectangle([4, 4, size[0]-5, size[1]-5], outline=tpl['border'], width=3)
    draw.rectangle([4, 4, size[0]-5, 40], fill=tpl['border'])
    draw.text((12, 12), tpl['header_text'], fill=(255, 255, 255), font=FONT_BOLD)

    # Photo box
    x0, y0, x1, y1 = tpl['photo_box']
    draw.rectangle([x0, y0, x1, y1], fill=(200, 200, 200), outline=tpl['border'], width=2)
    draw.text((x0 + 15, y0 + 35), 'PHOTO', fill=(120, 120, 120), font=FONT_SMALL)

    # Fields (labels + values in fixed boxes)
    fields = _synthetic_fields()
    y = tpl['field_y']
    for label in ['SURNAME', 'GIVEN NAMES', 'ID NO.', 'D.O.B.', 'EXPIRY', 'NATION.']:
        draw.text((tpl['field_x'], y), f'{label}:', fill=tpl['label_color'], font=FONT_SMALL)
        draw.text((tpl['value_x'], y), fields[label], fill=tpl['text'], font=FONT_REG)
        y += tpl['field_step']

    # Decorative line
    draw.line([15, 155, size[0]-15, 155], fill=tpl['border'], width=1)

    # MRZ-style bottom strip
    mrz = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789<', k=36))
    draw.text((15, tpl['mrz_y']), mrz[:30], fill=(70, 70, 70), font=FONT_MRZ)
    draw.text((15, tpl['mrz_y'] + 16), mrz[6:], fill=(70, 70, 70), font=FONT_MRZ)

    img = _apply_paper_texture(img, strength=4)
    return img


# ── Quality metrics (for stricter GOOD labels) ──────────────────────
QUALITY_THRESHOLDS = {
    'sharpness_min': 40.0,
    'contrast_min': 30.0,
    'noise_max': 12.0,
}


def quality_metrics(img):
    gray = np.array(img.convert('L'), dtype=np.float32)
    lap = cv2.Laplacian(gray, cv2.CV_32F)
    sharpness = float(lap.var())
    contrast = float(gray.std())
    blur_base = cv2.GaussianBlur(gray, (3, 3), 0)
    noise = float((gray - blur_base).std())
    return {
        'sharpness': sharpness,
        'contrast': contrast,
        'noise': noise,
    }


def quality_passes(img, thresholds=QUALITY_THRESHOLDS):
    m = quality_metrics(img)
    return (
        m['sharpness'] >= thresholds['sharpness_min'] and
        m['contrast']  >= thresholds['contrast_min'] and
        m['noise']     <= thresholds['noise_max']
    )


# ── Per-class augmentation pipelines ───────────────────────────────
def apply_good(img):
    """Minor realistic camera variations only (strictly usable)."""
    aug = A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.08,
                                   contrast_limit=0.08, p=0.5),
        A.GaussNoise(var_limit=(1, 6), p=0.25),
        A.Rotate(limit=2, p=0.25),
    ])
    return Image.fromarray(aug(image=np.array(img))['image'])


def apply_blurry(img):
    """Heavy motion/defocus blur — unmistakably blurry."""
    blur_type = random.choice(['gaussian', 'motion'])
    arr = np.array(img)
    if blur_type == 'gaussian':
        aug = A.GaussianBlur(blur_limit=(11, 21), p=1.0)
    else:
        aug = A.MotionBlur(blur_limit=(15, 25), p=1.0)
    blurred = aug(image=arr)['image']
    noise_aug = A.GaussNoise(var_limit=(20, 50), p=0.7)
    return Image.fromarray(noise_aug(image=blurred)['image'])


def apply_glare(img):
    """Bright specular reflection covering part of the document."""
    arr = np.array(img).copy()
    h, w = arr.shape[:2]

    cx = random.randint(w//4, 3*w//4)
    cy = random.randint(h//4, 3*h//4)
    rx = random.randint(w//5, w//2)
    ry = random.randint(h//5, h//2)

    for y in range(max(0, cy-ry), min(h, cy+ry)):
        for x in range(max(0, cx-rx), min(w, cx+rx)):
            dx = (x - cx) / rx
            dy = (y - cy) / ry
            dist = dx*dx + dy*dy
            if dist <= 1.0:
                intensity = 1.0 - dist
                arr[y, x] = np.clip(
                    arr[y, x] * (1 - intensity) + 255 * intensity,
                    0, 255
                ).astype(np.uint8)

    aug = A.RandomBrightnessContrast(
        brightness_limit=(0.3, 0.5), contrast_limit=(0.2, 0.4), p=1.0)
    return Image.fromarray(aug(image=arr)['image'])


def apply_dark(img):
    """Underexposed capture — clearly too dark to read."""
    aug = A.Compose([
        A.RandomBrightnessContrast(
            brightness_limit=(-0.65, -0.45),
            contrast_limit=(-0.2, 0.1), p=1.0),
        A.GaussNoise(var_limit=(15, 40), p=0.6),
    ])
    return Image.fromarray(aug(image=np.array(img))['image'])


def apply_no_document(size=(400, 260)):
    """Random non-document scene — table, hand, background, etc."""
    scene_type = random.choice(['solid', 'gradient', 'noise', 'hand_bg'])

    if scene_type == 'solid':
        color = tuple(random.randint(50, 220) for _ in range(3))
        img = Image.new('RGB', size, color)

    elif scene_type == 'gradient':
        arr = np.zeros((*size[::-1], 3), dtype=np.uint8)
        c1 = np.array([random.randint(30,180)] * 3)
        c2 = np.array([random.randint(80,240)] * 3)
        for i in range(size[1]):
            arr[i] = (c1 + (c2 - c1) * i / size[1]).astype(np.uint8)
        img = Image.fromarray(arr)

    elif scene_type == 'noise':
        arr = np.random.randint(60, 200, (*size[::-1], 3), dtype=np.uint8)
        img = Image.fromarray(
            A.GaussianBlur(blur_limit=(5,11), p=1.0)(image=arr)['image'])

    else:
        base = tuple(random.randint(140, 210) for _ in range(3))
        img = Image.new('RGB', size, base)
        draw = ImageDraw.Draw(img)
        for _ in range(random.randint(3, 8)):
            x0 = random.randint(0, size[0])
            y0 = random.randint(0, size[1])
            x1 = x0 + random.randint(20, 100)
            y1 = y0 + random.randint(20, 80)
            color = tuple(random.randint(100, 200) for _ in range(3))
            draw.ellipse([x0, y0, x1, y1], fill=color)
        img = img.filter(ImageFilter.GaussianBlur(radius=3))

    return img


# ── Dataset generation ──────────────────────────────────────────────
AUGMENT_FN = {
    'GOOD':        lambda img, _: apply_good(img),
    'BLURRY':      lambda img, _: apply_blurry(img),
    'GLARE':       lambda img, _: apply_glare(img),
    'DARK':        lambda img, _: apply_dark(img),
    'NO_DOCUMENT': lambda img, size: apply_no_document(size),
}


def generate_dataset(n_per_class=1000, output_dir=DATA_DIR, img_size=(400, 260)):
    """
    Generate n_per_class images per quality class.
    Total: n_per_class × 5 images.
    Default 1000/class = 5000 images total.
    """
    print(f'Generating {n_per_class} images per class ({n_per_class*5} total)...')
    metadata = []
    max_good_attempts = 8

    for cls in CLASSES:
        class_dir = output_dir / cls
        class_dir.mkdir(parents=True, exist_ok=True)
        print(f'  Generating class: {cls}', end='', flush=True)

        for i in range(n_per_class):
            attempts = 0
            while True:
                base = render_base_document(img_size)
                img = AUGMENT_FN[cls](base, img_size)

                if cls == 'GOOD' and not quality_passes(img):
                    attempts += 1
                    if attempts < max_good_attempts:
                        continue

                metrics = quality_metrics(img)
                img = img.resize((224, 224), Image.BILINEAR)
                break

            fname = f'{cls}_{i:05d}.jpg'
            fpath = class_dir / fname
            img.save(fpath, quality=90)

            metadata.append({
                'path':  str(fpath),
                'label': cls,
                'idx':   CLASS_TO_IDX[cls],
                'sharpness': metrics['sharpness'],
                'contrast': metrics['contrast'],
                'noise': metrics['noise'],
            })

            if (i+1) % 250 == 0:
                print(f' {i+1}', end='', flush=True)
        print(f' ✓')

    df = pd.DataFrame(metadata)
    df.to_csv(output_dir / 'metadata.csv', index=False)
    print(f'
Dataset saved to {output_dir}')
    print(f'Total samples: {len(df)}')
    return df

print('Generator defined — run next cell to generate dataset')


## Cell 5 — Generate Dataset

**Change `N_PER_CLASS`** to control dataset size:
- `500` → fast test run (~10 min training)
- `1000` → good quality (~20 min training) ← **recommended to start**
- `2000` → better generalisation (~40 min training)

In [ ]:
N_PER_CLASS = 1000  # ← change this

# Check if dataset already exists (avoid regenerating on reconnect)
metadata_path = DATA_DIR / 'metadata.csv'

if metadata_path.exists():
    df_meta = pd.read_csv(metadata_path)
    print(f'Dataset already exists: {len(df_meta)} samples')
    print(df_meta['label'].value_counts())
else:
    df_meta = generate_dataset(n_per_class=N_PER_CLASS)

# Visualise a sample from each class
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Sample Images Per Class', fontsize=14, fontweight='bold')
for ax, cls in zip(axes, CLASSES):
    sample = df_meta[df_meta['label'] == cls].sample(1).iloc[0]
    img = Image.open(sample['path'])
    ax.imshow(img)
    ax.set_title(cls, fontweight='bold', fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Sample images saved')

## Cell 6 — Train/Val/Test Split + PyTorch Dataset

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split — same class distribution in all splits
df_train, df_temp = train_test_split(
    df_meta, test_size=0.30, stratify=df_meta['label'], random_state=SEED)
df_val, df_test = train_test_split(
    df_temp,  test_size=0.50, stratify=df_temp['label'],  random_state=SEED)

print(f'Train:      {len(df_train):,} samples')
print(f'Validation: {len(df_val):,} samples')
print(f'Test:       {len(df_test):,} samples')
print(f'\nClass distribution (train):')
print(df_train['label'].value_counts())

# Save splits
df_train.to_csv(DATA_DIR / 'train.csv', index=False)
df_val.to_csv(DATA_DIR  / 'val.csv',   index=False)
df_test.to_csv(DATA_DIR / 'test.csv',  index=False)


# ── Transforms ──────────────────────────────────────────────────────
# Training: augmentation to increase robustness
TRAIN_TRANSFORM = A.Compose([
    A.RandomResizedCrop(height=224, width=224,
                        scale=(0.85, 1.0), p=1.0),
    A.HorizontalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15,
                               contrast_limit=0.15, p=0.4),
    A.Rotate(limit=5, p=0.3),
    A.GaussNoise(var_limit=(3, 12), p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Validation/Test: no augmentation, just normalize
EVAL_TRANSFORM = A.Compose([
    A.Resize(height=224, width=224),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])


# ── Dataset class ────────────────────────────────────────────────────
class QualityDataset(Dataset):
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = np.array(Image.open(row['path']).convert('RGB'))
        label = int(row['idx'])
        if self.transform:
            image = self.transform(image=image)['image']
        return image, label


# DataLoaders
BATCH_SIZE  = 64
NUM_WORKERS = 2

train_dataset = QualityDataset(df_train, TRAIN_TRANSFORM)
val_dataset   = QualityDataset(df_val,   EVAL_TRANSFORM)
test_dataset  = QualityDataset(df_test,  EVAL_TRANSFORM)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'\nDataLoaders ready')
print(f'Steps per epoch: {len(train_loader)}')

## Cell 7 — Build Model

**MobileNetV3-Small** pretrained on ImageNet.

We freeze the early layers (they already know edges, textures, gradients from
ImageNet) and only train:
- The last 2 convolutional blocks (learn document-specific features)
- The classification head (learn our 5 quality labels)

In [ ]:
def build_model(num_classes=5, freeze_early=True):
    """
    MobileNetV3-Small pretrained on ImageNet.
    Replaces the classification head with our 5-class head.
    """
    model = timm.create_model(
        'mobilenetv3_small_100',
        pretrained=True,
        num_classes=num_classes
    )

    if freeze_early:
        # Freeze all parameters first
        for param in model.parameters():
            param.requires_grad = False

        # Unfreeze last 2 blocks + classifier head
        # MobileNetV3-Small has blocks 0-10
        # We unfreeze blocks 9, 10, and the head
        for name, param in model.named_parameters():
            if any(layer in name for layer in ['blocks.9', 'blocks.10',
                                                'conv_head', 'classifier']):
                param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Model: MobileNetV3-Small')
    print(f'Total parameters:     {total:,}')
    print(f'Trainable parameters: {trainable:,} ({100*trainable/total:.1f}%)')

    return model


model = build_model(num_classes=NUM_CLASSES, freeze_early=True)
model = model.to(DEVICE)

# Quick sanity check
dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'\nOutput shape: {out.shape}  (expected: [2, 5])')
print('Model ready ✓')

## Cell 8 — Training Configuration

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────
CONFIG = {
    'model_name':     'mobilenetv3_small_100',
    'num_classes':    NUM_CLASSES,
    'input_size':     224,
    'batch_size':     BATCH_SIZE,
    'epochs':         30,
    'lr':             1e-4,
    'weight_decay':   1e-4,
    'label_smoothing': 0.1,   # prevents overconfidence
    'grad_clip':      1.0,
    'freeze_early':   True,
    'n_per_class':    N_PER_CLASS,
    'seed':           SEED,
}

# ── Loss, optimiser, scheduler ──────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay']
)

# Cosine annealing — LR decays smoothly from lr to ~0 over all epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['epochs'],
    eta_min=1e-6
)

print('Training config:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## Cell 9 — Training Loop

Logs every epoch to MLflow. Saves best model checkpoint to Drive.
Early stopping kicks in if val accuracy doesn't improve for 8 epochs.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping — prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), CONFIG['grad_clip'])

        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss    = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (total_loss / total, correct / total,
            np.array(all_preds), np.array(all_labels))


# ── Training run ─────────────────────────────────────────────────────
mlflow.set_experiment('kyc_quality_classifier')

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_acc  = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 8

CHECKPOINT_PATH = CHECKPOINT_DIR / 'quality_fp32_best.pth'

with mlflow.start_run(run_name='quality_fp32_baseline') as run:
    mlflow.log_params(CONFIG)
    print(f'MLflow run: {run.info.run_id[:8]}...')
    print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | '
          f'{"Val Loss":>8} | {"Val Acc":>7} | {"LR":>8}')
    print('-' * 65)

    for epoch in range(1, CONFIG['epochs'] + 1):
        t0 = time.time()

        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, optimizer, criterion)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader)

        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        train_losses.append(tr_loss)
        val_losses.append(vl_loss)
        train_accs.append(tr_acc)
        val_accs.append(vl_acc)

        mlflow.log_metrics({
            'train_loss': tr_loss, 'train_acc': tr_acc,
            'val_loss':   vl_loss, 'val_acc':   vl_acc,
            'lr':         current_lr
        }, step=epoch)

        elapsed = time.time() - t0
        print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>8.2%} | '
              f'{vl_loss:>8.4f} | {vl_acc:>6.2%} | '
              f'{current_lr:.2e}  ({elapsed:.0f}s)')

        # Save best model
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            patience_counter = 0
            torch.save({
                'epoch':      epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc':    vl_acc,
                'config':     CONFIG,
            }, CHECKPOINT_PATH)
            print(f'         → Best model saved (val_acc: {vl_acc:.2%})')
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f'\nEarly stopping at epoch {epoch} '
                      f'(no improvement for {EARLY_STOP_PATIENCE} epochs)')
                break

    mlflow.log_metric('best_val_acc', best_val_acc)
    print(f'\nTraining complete. Best val acc: {best_val_acc:.2%}')

## Cell 10 — Training Curves

In [ ]:
epochs_ran = len(train_losses)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Document Quality Classifier — Training Curves',
             fontsize=14, fontweight='bold')

ax1.plot(range(1, epochs_ran+1), train_losses, label='Train Loss', color='steelblue')
ax1.plot(range(1, epochs_ran+1), val_losses,   label='Val Loss',   color='coral')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(range(1, epochs_ran+1), [a*100 for a in train_accs],
         label='Train Acc', color='steelblue')
ax2.plot(range(1, epochs_ran+1), [a*100 for a in val_accs],
         label='Val Acc',   color='coral')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
ax2.axhline(y=90, color='green', linestyle='--', alpha=0.5, label='90% target')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Training curves saved')

## Cell 11 — Full Evaluation on Test Set

This is what goes in your thesis. Load best checkpoint, run on held-out test set.

In [ ]:
# Load best checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best checkpoint from epoch {checkpoint["epoch"]} '
      f'(val_acc: {checkpoint["val_acc"]:.2%})')

# Evaluate on test set
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader)

print(f'\n=== TEST SET RESULTS (FP32 Baseline) ===')
print(f'Test Accuracy: {test_acc:.2%}')
print(f'Test Loss:     {test_loss:.4f}')
print(f'\nPer-class Report:')
print(classification_report(test_labels, test_preds,
                             target_names=CLASSES, digits=4))

# F1 macro
f1_macro = f1_score(test_labels, test_preds, average='macro')
print(f'F1 Macro: {f1_macro:.4f}')

# Save metrics for thesis table
baseline_metrics = {
    'variant':    'FP32 Baseline',
    'test_acc':   round(test_acc, 4),
    'f1_macro':   round(f1_macro, 4),
    'test_loss':  round(test_loss, 4),
}

## Cell 12 — Confusion Matrix

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Document Quality Classifier — Confusion Matrix',
             fontsize=14, fontweight='bold')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax1)
ax1.set_title('Raw Counts')
ax1.set_ylabel('True Label')
ax1.set_xlabel('Predicted Label')

# Normalised
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax2)
ax2.set_title('Normalised (per true class)')
ax2.set_ylabel('True Label')
ax2.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# What to look for:
# - Diagonal should be bright (correct classifications)
# - Off-diagonal should be dark (few mistakes)
# - Common mistake: BLURRY confused with DARK (both degrade image quality)
print('Confusion matrix saved')

## Cell 13 — Model Size & Latency (FP32 Baseline)

This is your first thesis benchmark number.

In [ ]:
import os

def get_model_size_mb(model, path='/tmp/tmp_model.pth'):
    torch.save(model.state_dict(), path)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    os.remove(path)
    return round(size_mb, 2)

def measure_latency_ms(model, input_shape=(1, 3, 224, 224),
                        n_runs=200, device='cpu'):
    """
    Measure on CPU — this is what runs on the mobile device.
    Always measure on CPU for mobile benchmarking, even in Colab.
    """
    model_cpu = model.to('cpu').eval()
    dummy = torch.randn(input_shape)

    # Warmup
    with torch.no_grad():
        for _ in range(20):
            model_cpu(dummy)

    # Measure
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            model_cpu(dummy)
            times.append((time.perf_counter() - t0) * 1000)

    model.to(DEVICE)  # put back
    return {
        'mean_ms':   round(np.mean(times), 2),
        'std_ms':    round(np.std(times), 2),
        'p50_ms':    round(np.percentile(times, 50), 2),
        'p95_ms':    round(np.percentile(times, 95), 2),
    }

print('Measuring FP32 baseline...')
baseline_size = get_model_size_mb(model)
baseline_latency = measure_latency_ms(model)

print(f'\nFP32 Baseline Benchmarks:')
print(f'  Model size:      {baseline_size} MB')
print(f'  Latency (mean):  {baseline_latency["mean_ms"]} ms')
print(f'  Latency (p95):   {baseline_latency["p95_ms"]} ms')
print(f'  Test accuracy:   {test_acc:.2%}')

baseline_metrics.update({
    'size_mb':    baseline_size,
    'latency_mean_ms': baseline_latency['mean_ms'],
    'latency_p95_ms':  baseline_latency['p95_ms'],
})
print('\nNote: Colab CPU ≠ mobile CPU. '
      'Real device benchmarks done in Flutter (DOC3).')

## Cell 14 — Compression: INT8 Post-Training Quantization

3 lines of code. 4× smaller model. No retraining.

In [ ]:
import copy
from torch.quantization import quantize_dynamic

print('Applying INT8 Post-Training Quantization...')

# Load best checkpoint for clean PTQ
model_fp32_clean = build_model(num_classes=NUM_CLASSES, freeze_early=False)
model_fp32_clean.load_state_dict(
    torch.load(CHECKPOINT_PATH, map_location='cpu')['model_state_dict'])
model_fp32_clean.eval()

# ── Apply PTQ ────────────────────────────────────────────────────────
model_int8 = quantize_dynamic(
    model_fp32_clean,
    {nn.Linear, nn.Conv2d},   # quantize these layer types
    dtype=torch.qint8
)

print('Quantization applied. Evaluating...')

# Evaluate INT8 model on test set
model_int8.eval()
int8_preds, int8_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        # PTQ runs on CPU
        outputs = model_int8(images.cpu())
        preds   = outputs.argmax(dim=1)
        int8_preds.extend(preds.numpy())
        int8_labels.extend(labels.numpy())

int8_acc    = accuracy_score(int8_labels, int8_preds)
int8_f1     = f1_score(int8_labels, int8_preds, average='macro')
int8_size   = get_model_size_mb(model_int8, '/tmp/int8_model.pth')
int8_latency = measure_latency_ms(model_int8.cpu())

# Save INT8 checkpoint
torch.save(model_int8.state_dict(), CHECKPOINT_DIR / 'quality_int8_ptq.pth')

print(f'\nINT8 PTQ Results:')
print(f'  Size:       {int8_size} MB  (was {baseline_size} MB, '
      f'{baseline_size/int8_size:.1f}× reduction)')
print(f'  Latency:    {int8_latency["mean_ms"]} ms  '
      f'(was {baseline_latency["mean_ms"]} ms)')
print(f'  Accuracy:   {int8_acc:.2%}  '
      f'(was {test_acc:.2%}, delta: {(int8_acc-test_acc)*100:+.2f}pp)')
print(f'  F1 Macro:   {int8_f1:.4f}')

int8_metrics = {
    'variant':         'INT8 PTQ',
    'size_mb':         int8_size,
    'latency_mean_ms': int8_latency['mean_ms'],
    'latency_p95_ms':  int8_latency['p95_ms'],
    'test_acc':        round(int8_acc, 4),
    'f1_macro':        round(int8_f1, 4),
    'acc_delta_pp':    round((int8_acc - test_acc) * 100, 3),
}

## Cell 15 — Compression: Knowledge Distillation

Train a smaller student model to imitate the FP32 teacher.
Student is MobileNetV3-Small-050 (half the channels = ~4× smaller).

In [ ]:
class DistillationLoss(nn.Module):
    """
    Combined loss for knowledge distillation.

    total_loss = alpha × KL(student_soft, teacher_soft) × T²
               + (1 - alpha) × CrossEntropy(student, hard_labels)

    Temperature T: higher = softer teacher probabilities =
                   more information transferred to student
    Alpha: how much to weight the knowledge transfer vs. hard labels
    """
    def __init__(self, temperature=4.0, alpha=0.7):
        super().__init__()
        self.T     = temperature
        self.alpha = alpha
        self.ce    = nn.CrossEntropyLoss(label_smoothing=0.05)
        self.kl    = nn.KLDivLoss(reduction='batchmean')

    def forward(self, student_logits, teacher_logits, labels):
        # Hard label loss
        hard_loss = self.ce(student_logits, labels)

        # Soft label loss (knowledge transfer)
        student_soft = F.log_softmax(student_logits / self.T, dim=1)
        teacher_soft = F.softmax(teacher_logits   / self.T, dim=1)
        soft_loss    = self.kl(student_soft, teacher_soft) * (self.T ** 2)

        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss


# ── Build teacher (best FP32 model) ─────────────────────────────────
teacher = build_model(num_classes=NUM_CLASSES, freeze_early=False)
teacher.load_state_dict(
    torch.load(CHECKPOINT_PATH, map_location=DEVICE)['model_state_dict'])
teacher = teacher.to(DEVICE).eval()
# Freeze teacher completely
for p in teacher.parameters():
    p.requires_grad = False
print('Teacher loaded and frozen ✓')

# ── Build student (half channels) ───────────────────────────────────
student = timm.create_model(
    'mobilenetv3_small_050',   # 050 = 50% channel multiplier
    pretrained=True,
    num_classes=NUM_CLASSES
).to(DEVICE)
print(f'Student size: {get_model_size_mb(student)} MB')

# ── Distillation training ─────────────────────────────────────────
distill_loss = DistillationLoss(temperature=4.0, alpha=0.7)
distill_optimizer = torch.optim.AdamW(
    student.parameters(), lr=5e-4, weight_decay=1e-4)
distill_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    distill_optimizer, T_max=20, eta_min=1e-6)

DISTILL_EPOCHS = 20
best_distill_acc = 0.0
DISTILL_CHECKPOINT = CHECKPOINT_DIR / 'quality_distilled_best.pth'

print(f'\nDistillation training ({DISTILL_EPOCHS} epochs)...')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Val Acc":>8}')
print('-' * 35)

with mlflow.start_run(run_name='quality_distilled'):
    mlflow.log_params({
        'distill_temperature': 4.0,
        'distill_alpha': 0.7,
        'student_model': 'mobilenetv3_small_050',
    })

    for epoch in range(1, DISTILL_EPOCHS + 1):
        student.train()
        total_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            distill_optimizer.zero_grad()

            with torch.no_grad():
                teacher_logits = teacher(images)

            student_logits = student(images)
            loss = distill_loss(student_logits, teacher_logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            distill_optimizer.step()
            total_loss += loss.item()

        distill_scheduler.step()

        # Validate
        _, d_acc, _, _ = evaluate(student, val_loader)
        mlflow.log_metrics({'distill_loss': total_loss,
                            'distill_val_acc': d_acc}, step=epoch)

        print(f'{epoch:>6} | {total_loss:>10.4f} | {d_acc:>7.2%}')

        if d_acc > best_distill_acc:
            best_distill_acc = d_acc
            torch.save(student.state_dict(), DISTILL_CHECKPOINT)
            print(f'         → Saved (val_acc: {d_acc:.2%})')

print(f'\nDistillation complete. Best val_acc: {best_distill_acc:.2%}')

## Cell 16 — Evaluate Distilled Student

In [ ]:
# Load best distilled checkpoint
student.load_state_dict(torch.load(DISTILL_CHECKPOINT, map_location=DEVICE))

_, distill_acc, distill_preds, distill_test_labels = evaluate(student, test_loader)
distill_f1   = f1_score(distill_test_labels, distill_preds, average='macro')
distill_size = get_model_size_mb(student)
distill_latency = measure_latency_ms(student.cpu())
student.to(DEVICE)

print(f'Distilled Student Results:')
print(f'  Size:       {distill_size} MB  '
      f'(was {baseline_size} MB, {baseline_size/distill_size:.1f}× reduction)')
print(f'  Latency:    {distill_latency["mean_ms"]} ms  '
      f'(was {baseline_latency["mean_ms"]} ms)')
print(f'  Accuracy:   {distill_acc:.2%}  '
      f'(was {test_acc:.2%}, delta: {(distill_acc-test_acc)*100:+.2f}pp)')
print(f'  F1 Macro:   {distill_f1:.4f}')

distill_metrics = {
    'variant':         'Distilled Student',
    'size_mb':         distill_size,
    'latency_mean_ms': distill_latency['mean_ms'],
    'latency_p95_ms':  distill_latency['p95_ms'],
    'test_acc':        round(distill_acc, 4),
    'f1_macro':        round(distill_f1, 4),
    'acc_delta_pp':    round((distill_acc - test_acc) * 100, 3),
}

## Cell 17 — Thesis Compression Table

This table goes directly into your thesis. Copy it into your LaTeX or Word document.

In [ ]:
results_df = pd.DataFrame([
    baseline_metrics,
    int8_metrics,
    distill_metrics,
])

# Clean display
display_cols = ['variant', 'size_mb', 'latency_mean_ms',
                'test_acc', 'f1_macro', 'acc_delta_pp']
display_df = results_df[display_cols].copy()
display_df.columns = [
    'Variant', 'Size (MB)', 'Latency (ms)', 'Test Acc', 'F1 Macro', 'Acc Δ (pp)'
]
display_df['Test Acc'] = display_df['Test Acc'].map('{:.2%}'.format)
display_df['F1 Macro'] = display_df['F1 Macro'].map('{:.4f}'.format)

print('=== THESIS TABLE: Document Quality Model — Compression Study ===')
print(display_df.to_string(index=False))

# Save
results_df.to_csv(RESULTS_DIR / 'compression_results.csv', index=False)

# Visual bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Document Quality Model — Compression Study',
             fontsize=14, fontweight='bold')

variants = [r['variant'] for r in [baseline_metrics, int8_metrics, distill_metrics]]
colors   = ['#2196F3', '#FF9800', '#4CAF50']

# Size
sizes = [r['size_mb'] for r in [baseline_metrics, int8_metrics, distill_metrics]]
bars = axes[0].bar(variants, sizes, color=colors)
axes[0].set_title('Model Size (MB)'); axes[0].set_ylabel('MB')
for bar, val in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val}', ha='center', fontsize=11, fontweight='bold')

# Latency
lats = [r['latency_mean_ms'] for r in
        [baseline_metrics, int8_metrics, distill_metrics]]
bars = axes[1].bar(variants, lats, color=colors)
axes[1].set_title('Inference Latency (ms, CPU)'); axes[1].set_ylabel('ms')
for bar, val in zip(bars, lats):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val}', ha='center', fontsize=11, fontweight='bold')

# Accuracy
accs = [r['test_acc']*100 for r in
        [baseline_metrics, int8_metrics, distill_metrics]]
bars = axes[2].bar(variants, accs, color=colors)
axes[2].set_title('Test Accuracy (%)'); axes[2].set_ylabel('%')
axes[2].set_ylim(min(accs) - 5, 100)
for bar, val in zip(bars, accs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'compression_comparison.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('Results saved')

## Cell 18 — Export to ONNX

Both the FP32 model and distilled student are exported.

In [ ]:
import onnx
import onnxruntime as ort

def export_to_onnx(model, output_path, model_name='model', input_size=224):
    model_cpu = model.to('cpu').eval()
    dummy     = torch.randn(1, 3, input_size, input_size)

    torch.onnx.export(
        model_cpu, dummy, str(output_path),
        opset_version=12,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={
            'input':  {0: 'batch_size'},
            'output': {0: 'batch_size'}
        },
        export_params=True,
    )

    # Validate exported model
    onnx_model = onnx.load(str(output_path))
    onnx.checker.check_model(onnx_model)

    # Run inference check
    ort_session = ort.InferenceSession(str(output_path))
    ort_out = ort_session.run(None, {'input': dummy.numpy()})

    onnx_size = os.path.getsize(output_path) / (1024 * 1024)
    print(f'✓ {model_name} ONNX exported: {output_path.name} '
          f'({onnx_size:.2f} MB)')
    print(f'  Output shape: {ort_out[0].shape}  '
          f'(expected: [1, {NUM_CLASSES}])')
    model.to(DEVICE)
    return ort_session

# Export FP32 baseline
model_fp32_export = build_model(num_classes=NUM_CLASSES, freeze_early=False)
model_fp32_export.load_state_dict(
    torch.load(CHECKPOINT_PATH, map_location='cpu')['model_state_dict'])
export_to_onnx(model_fp32_export,
               EXPORT_DIR / 'doc_quality_fp32.onnx', 'FP32')

# Export distilled student
student_export = timm.create_model(
    'mobilenetv3_small_050', pretrained=False, num_classes=NUM_CLASSES)
student_export.load_state_dict(
    torch.load(DISTILL_CHECKPOINT, map_location='cpu'))
export_to_onnx(student_export,
               EXPORT_DIR / 'doc_quality_distilled.onnx', 'Distilled')

## Cell 19 — Export to TFLite (for Flutter)

This creates the `.tflite` file that goes into `assets/models/` in the Flutter app.

In [ ]:
import subprocess

def onnx_to_tflite(onnx_path, tflite_path, quantize_int8=False):
    """
    Convert ONNX model to TFLite format.
    Step 1: ONNX → TensorFlow SavedModel (via onnx-tf)
    Step 2: SavedModel → TFLite (via tf.lite.TFLiteConverter)
    """
    import tensorflow as tf

    saved_model_dir = str(tflite_path).replace('.tflite', '_saved_model')

    # Step 1: ONNX → SavedModel
    print(f'Converting {onnx_path.name} → SavedModel...')
    result = subprocess.run([
        'python', '-m', 'onnx_tf.main', 'convert',
        '-i', str(onnx_path),
        '-o', saved_model_dir
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print(f'Conversion error: {result.stderr[:500]}')
        return None

    # Step 2: SavedModel → TFLite
    print(f'Converting SavedModel → TFLite...')
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)

    if quantize_int8:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        print('  Applying INT8 optimisation...')

    tflite_model = converter.convert()

    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)

    tflite_size = os.path.getsize(tflite_path) / (1024 * 1024)
    print(f'✓ TFLite exported: {tflite_path.name} ({tflite_size:.2f} MB)')

    # Validate
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    print(f'  Input:  {input_details[0]["shape"]}  '
          f'dtype={input_details[0]["dtype"].__name__}')
    print(f'  Output: {output_details[0]["shape"]}  '
          f'dtype={output_details[0]["dtype"].__name__}')

    return tflite_path

# Export FP32 → TFLite
onnx_to_tflite(
    EXPORT_DIR / 'doc_quality_fp32.onnx',
    EXPORT_DIR / 'doc_quality.tflite',
    quantize_int8=False
)

# Export with INT8 TFLite optimisation
onnx_to_tflite(
    EXPORT_DIR / 'doc_quality_fp32.onnx',
    EXPORT_DIR / 'doc_quality_int8.tflite',
    quantize_int8=True
)

print('\nFiles ready for Flutter:')
print(f'  Primary:   {EXPORT_DIR}/doc_quality.tflite')
print(f'  INT8:      {EXPORT_DIR}/doc_quality_int8.tflite')
print('\nCopy these to: kyc_flutter/assets/models/')

## Post-MVP Hardening (Startup Readiness)

These are the two minimum additions before production use:

1) **Failure analysis**
- Collect 200–500 real device captures
- Run the model and log top failure modes
- Add a short error taxonomy (blur, glare, framing, doc type)

2) **Basic retraining loop**
- Save misclassified samples + metadata
- Retrain monthly or per release
- Track accuracy drift over time


## Cell 20 — Final Summary

Everything saved to Google Drive. Summary of all results.

In [ ]:
print('=' * 60)
print('DOCUMENT QUALITY CLASSIFIER — COMPLETE SUMMARY')
print('=' * 60)

print(f'\nDataset:')
print(f'  Total samples: {len(df_meta):,}')
print(f'  Samples/class: {N_PER_CLASS}')
print(f'  Train/Val/Test: {len(df_train)}/{len(df_val)}/{len(df_test)}')

print(f'\nCompression Study Results:')
print(display_df.to_string(index=False))

print(f'\nFiles saved to Drive:')
for f in sorted(CHECKPOINT_DIR.glob('quality_*')):
    size = os.path.getsize(f) / (1024 * 1024)
    print(f'  {f.name:40s} {size:.1f} MB')
for f in sorted(EXPORT_DIR.glob('doc_quality*')):
    size = os.path.getsize(f) / (1024 * 1024)
    print(f'  {f.name:40s} {size:.1f} MB')

print(f'\nNext steps:')
print('  1. Copy doc_quality.tflite to kyc_flutter/assets/models/')
print('  2. Open KYC_02_Face_Embedding.ipynb for Model 3')
print('  3. Open KYC_03_Liveness_Detection.ipynb for Model 4')

# Save full summary JSON
summary = {
    'model': 'DocumentQualityClassifier',
    'architecture': 'MobileNetV3-Small',
    'classes': CLASSES,
    'n_per_class': N_PER_CLASS,
    'results': [baseline_metrics, int8_metrics, distill_metrics]
}
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('\nSummary saved to Drive ✓')